# Using Word Embeddings to Improve GenAI Prompts

**Aim:** Use word embeddings to retrieve words similar to the keywords in a prompt, use those similar words to enrich the prompt, generate responses for both the original and enriched prompts with a GenAI model, and compare the two outputs for detail and relevance.

**Pipeline:**
1. Load a pre-trained word embedding model (GloVe, via `gensim`).
2. Extract keywords (nouns/adjectives) from the original prompt.
3. Retrieve the most similar words for each keyword using cosine similarity in embedding space.
4. Append the similar words to the original prompt to build an **enriched prompt**.
5. Feed both prompts to a text-generation model (GPT-2 via `transformers`) and generate responses.
6. Compare the two responses (word count, vocabulary richness, new words introduced).

## Step 0: Install dependencies
Run this cell once (needs internet access to download the packages and, later, the GloVe vectors and GPT-2 weights).

In [ ]:
!pip install -q gensim transformers torch nltk

## Step 1: Imports and NLTK setup

In [ ]:
import re
import nltk
import gensim.downloader as api
from transformers import pipeline, set_seed

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

## Step 2: Load a pre-trained word embedding model
We use GloVe (`glove-wiki-gigaword-50`), a 50-dimensional word embedding trained on Wikipedia + Gigaword. `gensim`'s downloader caches it locally after the first run, so later runs are fast.

In [ ]:
print("Loading word embedding model (GloVe, 50-dim)... this can take a minute the first time.")
word_vectors = api.load("glove-wiki-gigaword-50")
print("Word embedding model loaded. Vocabulary size:", len(word_vectors.index_to_key))

## Step 3: Retrieve similar words using embeddings
`most_similar` returns the words whose embedding vectors are closest (by cosine similarity) to the given word's vector.

In [ ]:
def get_similar_words(word, topn=3):
    """Return up to `topn` words most similar to `word` using GloVe embeddings."""
    word = word.lower()
    try:
        similar = word_vectors.most_similar(word, topn=topn)
        return [w for w, score in similar]
    except KeyError:
        # word not in the embedding vocabulary
        return []

# quick demo
print("Similar to 'energy':", get_similar_words("energy"))
print("Similar to 'climate':", get_similar_words("climate"))

## Step 4: Extract keywords from the prompt
We keep it simple: POS-tag the prompt and pull out nouns and adjectives, since those carry most of the topical meaning.

In [ ]:
def extract_keywords(prompt):
    """Pick out nouns/adjectives from the prompt as candidate keywords."""
    tokens = nltk.word_tokenize(prompt)
    tagged = nltk.pos_tag(tokens)
    keywords = [word for word, tag in tagged if tag.startswith('NN') or tag.startswith('JJ')]
    return list(dict.fromkeys(keywords))  # de-duplicate, keep order

print(extract_keywords("Write a short note about renewable energy."))

## Step 5: Enrich the prompt using similar words
For each keyword, fetch its nearest neighbours in embedding space and append them to the prompt as extra context/guidance for the generator.

In [ ]:
def enrich_prompt(prompt, topn=3):
    keywords = extract_keywords(prompt)
    enrichment_terms = set()
    for kw in keywords:
        enrichment_terms.update(get_similar_words(kw, topn=topn))
    # don't repeat words that are already in the prompt
    enrichment_terms -= set(k.lower() for k in keywords)
    if not enrichment_terms:
        return prompt
    enriched = (
        f"{prompt} "
        f"(Consider also related concepts such as: {', '.join(sorted(enrichment_terms))}.)"
    )
    return enriched

original_prompt = "Write a short note about renewable energy."
enriched_prompt = enrich_prompt(original_prompt)

print("Original Prompt:\n", original_prompt)
print("\nEnriched Prompt:\n", enriched_prompt)

## Step 6: Load a GenAI text-generation model
This notebook uses GPT-2 (via `transformers`) so it runs end-to-end with no API key. If you'd rather use the OpenAI API, see the **Optional: use OpenAI API** cell below instead of running this one.

In [ ]:
generator = pipeline("text-generation", model="gpt2")
set_seed(42)

def generate_response(prompt, max_length=120):
    output = generator(prompt, max_length=max_length, num_return_sequences=1, truncation=True)
    return output[0]['generated_text']

### Optional: use the OpenAI API instead of GPT-2
If your lab wants outputs from a hosted model (e.g. GPT-3.5/4) rather than local GPT-2, swap `generate_response` for this version. Requires `pip install openai` and an API key.

```python
from openai import OpenAI
client = OpenAI(api_key="YOUR_API_KEY")

def generate_response(prompt, max_length=200):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_length,
    )
    return response.choices[0].message.content
```

## Step 7: Generate responses for both prompts

In [ ]:
original_response = generate_response(original_prompt)
enriched_response = generate_response(enriched_prompt)

print("=== Original Prompt Response ===\n")
print(original_response)

print("\n\n=== Enriched Prompt Response ===\n")
print(enriched_response)

## Step 8: Compare the outputs
We compare the two responses on a few simple, objective measures: total word count, vocabulary size (unique words), and how many *new* words the enriched response introduces relative to the original.

In [ ]:
def compare_outputs(resp1, resp2):
    words1 = set(re.findall(r'\w+', resp1.lower()))
    words2 = set(re.findall(r'\w+', resp2.lower()))
    print(f"Original response word count       : {len(resp1.split())}")
    print(f"Enriched response word count        : {len(resp2.split())}")
    print(f"Unique words in original response   : {len(words1)}")
    print(f"Unique words in enriched response   : {len(words2)}")
    print(f"New words introduced by enrichment  : {len(words2 - words1)}")
    print(f"New words: {sorted(words2 - words1)}")

compare_outputs(original_response, enriched_response)

## Observations / Conclusion

- The **enriched prompt** injects semantically related terms (retrieved via word-embedding similarity) directly into the prompt text, giving the generator extra topical context to draw on.
- In most runs, the enriched response covers a **wider range of sub-topics** and uses a **richer vocabulary** than the response to the bare original prompt, since the model has more cues about what related concepts to touch on.
- The trade-off is that if the embedding model returns loosely-related or noisy neighbours (common with small models like `glove-wiki-gigaword-50`), the enriched prompt can drift slightly off-topic — using a larger embedding model (e.g. `glove-wiki-gigaword-300` or `word2vec-google-news-300`) or filtering similar words by a similarity-score threshold improves relevance further.
- This experiment can be repeated with different `original_prompt` values and different `topn` settings to study how the amount of injected context affects response detail and relevance.